# Heart Disease Prediction



### Loading the data


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/cleveland.csv', na_values='?').dropna()
df['disease'] = df['num'].apply(lambda x: True if x > 0 else False)

In [24]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


### Preparing the features


In [18]:
numerical = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
binary = ['sex', 'fbs', 'exang']
categorical = ['cp', 'restecg', 'slope', 'thal']
attributes = numerical + binary + categorical

# One-hot encode the categorical attributes
feat_df = pd.get_dummies(df[attributes], columns=categorical, dtype=float)

# Remember which columns belong to each attribute (e.g. cp -> cp_1.0, cp_2.0, ...)
columns_for = {a: [c for c in feat_df.columns if c == a or c.startswith(a + '_')] for a in attributes}

x_train, x_test, y_train, y_test = train_test_split(feat_df, df['disease'], test_size=0.2, random_state=63)

# Standardize features using only the training set's mean and std
for feature in numerical:
    mean, std = x_train[feature].mean(), x_train[feature].std()
    x_train[feature] = (x_train[feature] - mean) / std
    x_test[feature] = (x_test[feature] - mean) / std

### Our own kNN
We aren't allowed to use sklearn's `KNeighborsClassifier`, so this small class does the same job: it finds the k closest training patients with `NearestNeighbors` and goes with the majority vote. Writing it this way means `GridSearchCV` can still use it.

In [25]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import NearestNeighbors

class MyKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.fit_ = NearestNeighbors(n_neighbors=self.n_neighbors).fit(np.asarray(X))
        self.y_ = np.asarray(y)
        self.classes_ = np.unique(self.y_)
        return self

    def predict(self, X):
        distances, indices = self.fit_.kneighbors(np.asarray(X))
        predictions = []
        for neighbors in indices:
            disease = 0
            healthy = 0
            for index in neighbors:
                if self.y_[index]:
                    disease += 1
                else:
                    healthy += 1
            predictions.append(disease > healthy)
        return np.array(predictions)

### Baseline: all 13 attributes
To get a starting point, we throw in every attribute and let `GridSearchCV` try each odd k from 1 to 41. For every k it runs 10-fold cross-validation on the training set and averages the F1 scores.

In [26]:
from sklearn.model_selection import GridSearchCV, KFold

kfold = KFold(n_splits=10, shuffle=True, random_state=63)
nn = MyKNN()
params = {'n_neighbors': list(range(1, 42, 2))}

features = [c for a in attributes for c in columns_for[a]]
clf = GridSearchCV(nn, params, cv=kfold, scoring='f1')
clf.fit(x_train[features], y_train)
print(f"Results for all {len(attributes)} attributes: Best params: {clf.best_params_}, Best score: {clf.best_score_}")

Results for all 13 attributes: Best params: {'n_neighbors': 7}, Best score: 0.7998226307974592


### Picking the attributes
Using everything isn't necessarily best, since weak attributes just add noise to the distances. So we build the set up one attribute at a time: at each step we try adding every remaining attribute, run the same grid search over k, and keep whichever one helps the most. At the end we pick the step with the highest F1.

In [27]:
selected = []
remaining = list(attributes)
history = []
while remaining:
    best_score, best_attribute, best_k = 0, None, None
    for attribute in remaining:
        features = [c for a in selected + [attribute] for c in columns_for[a]]
        clf = GridSearchCV(nn, params, cv=kfold, scoring='f1')
        clf.fit(x_train[features], y_train)
        if clf.best_score_ > best_score:
            best_score, best_attribute, best_k = clf.best_score_, attribute, clf.best_params_['n_neighbors']
    selected.append(best_attribute)
    remaining.remove(best_attribute)
    history.append((best_score, best_k, list(selected)))
    print(f"Results for {len(selected)} attributes: Best k: {best_k}, Best score: {best_score:.4f}, Attributes: {selected}")

best_score, k, chosen = max(history, key=lambda h: h[0])
features = [c for a in chosen for c in columns_for[a]]
print(f"\nChosen: k = {k}, attributes = {chosen}, cross-validated F1 = {best_score:.4f}")

Results for 1 attributes: Best k: 29, Best score: 0.7431, Attributes: ['thal']
Results for 2 attributes: Best k: 39, Best score: 0.7836, Attributes: ['thal', 'ca']
Results for 3 attributes: Best k: 23, Best score: 0.8080, Attributes: ['thal', 'ca', 'cp']
Results for 4 attributes: Best k: 21, Best score: 0.8316, Attributes: ['thal', 'ca', 'cp', 'slope']
Results for 5 attributes: Best k: 13, Best score: 0.8445, Attributes: ['thal', 'ca', 'cp', 'slope', 'exang']
Results for 6 attributes: Best k: 13, Best score: 0.8474, Attributes: ['thal', 'ca', 'cp', 'slope', 'exang', 'fbs']
Results for 7 attributes: Best k: 41, Best score: 0.8539, Attributes: ['thal', 'ca', 'cp', 'slope', 'exang', 'fbs', 'trestbps']
Results for 8 attributes: Best k: 15, Best score: 0.8425, Attributes: ['thal', 'ca', 'cp', 'slope', 'exang', 'fbs', 'trestbps', 'chol']
Results for 9 attributes: Best k: 9, Best score: 0.8330, Attributes: ['thal', 'ca', 'cp', 'slope', 'exang', 'fbs', 'trestbps', 'chol', 'age']
Results for 10

### Final results on the test set
Now that we know which k and attributes to use, we do the 10-fold cross-validation the assignment asks for. Each fold gives us a slightly different model (trained on 9/10 of the training set), and we test every one of those 10 models on the held-out test set to get its precision, recall and F1.

In [22]:
kfold = KFold(n_splits=10, shuffle=True, random_state=63)
precision_list = []
recall_list = []
f1_score_list = []
for i, (train_index, val_index) in enumerate(kfold.split(x_train, y_train)):
    print(f"Fold {i+1}:")
    x_df = x_train.iloc[train_index]
    x = x_df[features].values
    nn = NearestNeighbors(n_neighbors=k)
    fit = nn.fit(x)

    confusion_matrix = {
        "TP": 0,
        "TN": 0,
        "FP": 0,
        "FN": 0
    }

    for j in range(len(x_test)):
        distances, indices = fit.kneighbors([x_test.iloc[j][features].values])
        disease = 0
        healthy = 0
        for index in indices[0]:
            if y_train.iloc[train_index[index]]:
                disease += 1
            else:
                healthy += 1
        if disease > healthy:
            if y_test.iloc[j]:
                confusion_matrix["TP"] += 1
            else:
                confusion_matrix["FP"] += 1
        else:
            if y_test.iloc[j]:
                confusion_matrix["FN"] += 1
            else:
                confusion_matrix["TN"] += 1

    print(f"Confusion Matrix: {confusion_matrix}")
    precision = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FP"]) if (confusion_matrix["TP"] + confusion_matrix["FP"]) > 0 else 0
    recall = confusion_matrix["TP"] / (confusion_matrix["TP"] + confusion_matrix["FN"]) if (confusion_matrix["TP"] + confusion_matrix["FN"]) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}")

    precision_list.append(precision)
    recall_list.append(recall)
    f1_score_list.append(f1_score)

    print()

print(f"Average Precision: {sum(precision_list)/len(precision_list):.4f}")
print(f"Average Recall: {sum(recall_list)/len(recall_list):.4f}")
print(f"Average F1 Score: {sum(f1_score_list)/len(f1_score_list):.4f}")

Fold 1:
Confusion Matrix: {'TP': 22, 'TN': 30, 'FP': 7, 'FN': 1}
Precision: 0.7586, Recall: 0.9565, F1 Score: 0.8462

Fold 2:
Confusion Matrix: {'TP': 22, 'TN': 31, 'FP': 6, 'FN': 1}
Precision: 0.7857, Recall: 0.9565, F1 Score: 0.8627

Fold 3:
Confusion Matrix: {'TP': 22, 'TN': 32, 'FP': 5, 'FN': 1}
Precision: 0.8148, Recall: 0.9565, F1 Score: 0.8800

Fold 4:
Confusion Matrix: {'TP': 23, 'TN': 29, 'FP': 8, 'FN': 0}
Precision: 0.7419, Recall: 1.0000, F1 Score: 0.8519

Fold 5:
Confusion Matrix: {'TP': 21, 'TN': 31, 'FP': 6, 'FN': 2}
Precision: 0.7778, Recall: 0.9130, F1 Score: 0.8400

Fold 6:
Confusion Matrix: {'TP': 21, 'TN': 32, 'FP': 5, 'FN': 2}
Precision: 0.8077, Recall: 0.9130, F1 Score: 0.8571

Fold 7:
Confusion Matrix: {'TP': 21, 'TN': 32, 'FP': 5, 'FN': 2}
Precision: 0.8077, Recall: 0.9130, F1 Score: 0.8571

Fold 8:
Confusion Matrix: {'TP': 22, 'TN': 33, 'FP': 4, 'FN': 1}
Precision: 0.8462, Recall: 0.9565, F1 Score: 0.8980

Fold 9:
Confusion Matrix: {'TP': 22, 'TN': 31, 'FP': 6, 

### Summary
- **k = 41**
- **Attributes:** thal, ca, cp, slope, exang, fbs, trestbps
- **Mean F1 on the test set: 0.8636** (average precision 0.7941, average recall 0.9478)

Picking attributes clearly paid off. Using all 13 only got a cross-validated F1 of about 0.80, while our 7 attributes reached about 0.85. The model leans towards saying "disease": it catches almost every sick patient (high recall) but flags a few healthy ones too (lower precision). For a screening tool that's probably the better way to be wrong. k = 41 is the top of the range we searched, so a slightly larger k might do as well or better.